In [1]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [2]:
import datetime as dt
import yfinance as yf


In [3]:
start = dt.datetime(2013,6,1)
end = dt.datetime(2022,2,11)
stk_data = yf.Ticker('TATACONSUM.NS').history(start=start, end=end)

In [4]:
stk_data.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2013-06-03 00:00:00+05:30,123.533149,125.649893,122.178441,124.506859,1446953,0.0,0.0
2013-06-04 00:00:00+05:30,125.141880,128.867344,124.549195,126.919937,3464825,0.0,0.0
2013-06-05 00:00:00+05:30,127.512652,128.613349,126.242606,127.173965,1965493,0.0,0.0
2013-06-06 00:00:00+05:30,126.750609,127.724319,124.803202,126.665947,1633085,0.0,0.0
2013-06-07 00:00:00+05:30,126.157909,129.502361,125.988572,126.242584,1955532,0.0,0.0


In [5]:
stk_data=stk_data[["Open","High","Low","Close"]]

In [6]:
from sklearn.preprocessing import MinMaxScaler
Ms = MinMaxScaler()
data1= Ms.fit_transform(stk_data)
print("Len:",data1.shape)

Len: (2144, 4)


In [7]:
data1=pd.DataFrame(data1,columns=["Open","High","Low","Close"])

In [8]:
training_size = round(len(data1 ) * 0.80)
print(training_size)
X_train=data1[:training_size]
X_test=data1[training_size:]
print("X_train length:",X_train.shape)
print("X_test length:",X_test.shape)
y_train=data1[:training_size]
y_test=data1[training_size:]
print("y_train length:",y_train.shape)
print("y_test length:",y_test.shape)

1715
X_train length: (1715, 4)
X_test length: (429, 4)
y_train length: (1715, 4)
y_test length: (429, 4)


In [9]:
performance_metrics={"Model":[],"RMSE":[],"MaPe":[],"Lag":[],"Test":[]}

In [10]:
def cominbation(dataset,listt):
    print(listt)
    datasetTwo=dataset[listt]
    test_obs = 28
    train =datasetTwo[:-test_obs]
    test = datasetTwo[-test_obs:]
    from statsmodels.tsa.api import VAR
    for i in [1,2,3,4,5,6,7,8,9,10]:
        model = VAR(train)
        results = model.fit(i)
        print('Order =', i)
        print('AIC: ', results.aic)
        print('BIC: ', results.bic)
        print()
    x = model.select_order(maxlags=12)
    order=x.selected_orders["aic"]
    result = model.fit(order)
    #result.summary()
    lagged_Values = train.values[-order:]
    pred = result.forecast(y=lagged_Values,steps=28) 
    preds=pd.DataFrame(pred,columns=listt)
    preds.to_csv("varforecasted_{}.csv".format(test_obs))
    from sklearn.metrics import mean_squared_error
    rmse= round(mean_squared_error(test,pred))
    from sklearn.metrics import mean_absolute_percentage_error
    mape=mean_absolute_percentage_error(test,pred)
    performance_metrics["Model"].append(listt)
    performance_metrics["RMSE"].append(rmse)
    performance_metrics["MaPe"].append(mape)
    performance_metrics["Lag"].append(order)
    performance_metrics["Test"].append(test_obs)
    perf=pd.DataFrame(performance_metrics)
    return perf,result,pred

In [11]:
listt=["Close","High","Open","Low"]

In [12]:
perf,result,pred=cominbation(data1,listt)

['Close', 'High', 'Open', 'Low']
Order = 1
AIC:  -44.50374353559928
BIC:  -44.45025124159

Order = 2
AIC:  -44.576675752010814
BIC:  -44.480352129498286

Order = 3
AIC:  -44.59354820054175
BIC:  -44.45435987639786

Order = 4
AIC:  -44.61213836839979
BIC:  -44.4300519271838

Order = 5
AIC:  -44.63326522848158
BIC:  -44.40824721236634

Order = 6
AIC:  -44.65075449015395
BIC:  -44.38277139885205

Order = 7
AIC:  -44.6499113555602
BIC:  -44.338929646249845

Order = 8
AIC:  -44.66234860267072
BIC:  -44.30833468992153

Order = 9
AIC:  -44.66280368962613
BIC:  -44.265723945324744

Order = 10
AIC:  -44.66786132612039
BIC:  -44.22768207939589



In [13]:
data1.head()

,Open,High,Low,Close
0,0.042448,0.043230,0.044432,0.044287
1,0.044571,0.047463,0.047579,0.047482
2,0.047699,0.047128,0.049827,0.047818
3,0.046694,0.045959,0.047916,0.047146
4,0.045912,0.048298,0.049490,0.046585


In [14]:
perf

,Model,RMSE,MaPe,Lag,Test
0,"[Close, High, Open, Low]",0,0.033589,11,28
